In [3]:
import os
import pdfplumber
import chromadb

from dotenv import load_dotenv
from google import genai


In [27]:
client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [4]:
db_client = chromadb.PersistentClient(
    path= "./chroma_db"
)

In [32]:
try:
    collection = db_client.get_collection(
        name = "multimodal_rag"
    )
    print("collection loaded")
except:
    collection = db_client.create_collection(
        name="multimodal_rag"
    )
    print("collection created")

collection created


In [17]:
def load_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()
    if extension ==".txt":
        with open(file_path,"r",encoding="utf-8") as file:
            return file.read()
    
    elif extension ==".pdf":
        text =""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
            
                if page_text:
                    text += page_text+ "\n"
        text = text.replace("\u2028", "\n")
        text = text.replace("\u2029", "\n")
        return text
    else:
        raise ValueError(f"Unsupported file extension {extension}")


In [18]:
def chunk_text(text,chunk_size=1000,overlap=200):
    chunks = []
    start =0
    while start<len(text):
        end = start+chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks    

In [23]:
def load_folder(folder_path):
    all_chunks = []
    for file in os.listdir(folder_path):
        file_path = os.path.join(
            folder_path,
            file
        )
        print(f"Reading file {file}")
        text = load_document(file_path)
        chunks = chunk_text(text)
        for i,chunk in enumerate(chunks):
            all_chunks.append(
                {
                    "source":file,
                    "chunk_id": i,
                    "content": chunk
                }
            )
    return all_chunks

In [24]:
all_chunks = load_folder("../Data")

Reading file M1.pdf
Reading file MLDOC.txt


In [29]:
def create_embedding(text):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents = text
    )
    return response.embeddings[0].values

In [30]:
# embedding = create_embedding(
#     all_chunks[0]["content"]
# )

# print(len(embedding))

3072


In [35]:

# for i, chunk in enumerate(all_chunks):
#     embedding = create_embedding(chunk["content"])
#     collection.add(
#         ids = [str(i)],
#         documents = [chunk["content"]],
#         embeddings = [embedding],
#         metadatas=[
#             {
#                 "source" : chunk["source"],
#                 "chunk_id" :chunk["chunk_id"]
#             }
#         ]
#        )
# print("stored successfully")


stored successfully


In [39]:
def ask_question(query):
    
    # Step 1: Create query embedding

    query_embedding = create_embedding(query)

    # Step 2: Retrieve top 3 relevant chunks
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    # Step 3: Extract retrieved chunks
    retrieved_chunks = results["documents"][0]

    # Step 4: Create context
    context = "\n\n".join(retrieved_chunks)

    # Step 5: Create prompt
    prompt = f"""
    You are a helpful AI assistant.

    Answer ONLY from the provided context.

    If the answer is not present in the context,
    say "Answer not found in document."

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    # Step 6: Generate answer
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return {
        "answer": response.text,
        "sources": results["metadatas"][0]
    }

In [40]:
query = input("Enter a Question")
Answer = ask_question(query)
print(Answer["answer"])
print("\nSources:")
for source in Answer["sources"]:
    print(source)

Supervised learning, also known as supervised machine learning, is defined by its use of labeled datasets to train algorithms to classify data or predict outcomes accurately. As input data is fed into the model, the model adjusts its weights until it has been fitted appropriately. This occurs as part of the cross validation process to ensure that the model avoids overfitting or underfitting.

Sources:
{'chunk_id': 7, 'source': 'MLDOC.txt'}
{'source': 'MLDOC.txt', 'chunk_id': 6}
{'chunk_id': 9, 'source': 'MLDOC.txt'}
{'source': 'MLDOC.txt', 'chunk_id': 3}
{'chunk_id': 8, 'source': 'MLDOC.txt'}
